# Lab 2 — Two providers, one interface

*Day 1, hour 5 · 50 minutes · pairs*

::: {.callout-note appearance="simple"}
**Objective** — run the one contract suite against every adapter, serve the
open-weight route with no code change, measure time-to-first-token, survive a live
429 storm, and record the first provider-comparison table.

**Before you start** — Module 1's lab complete. Gateway running.

**You finish with** — contract tests green on every adapter, a fault-drill log
excerpt, and `BENCHMARKS.md` seeded with numbers you produced.
:::

In [1]:
import os, pathlib, sys, re, subprocess, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + "\[[0-9;]*m")

for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (cand / "src" / "murshid").is_dir():
        os.chdir(cand); break
    if (cand / "murshid" / "src" / "murshid").is_dir():
        os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

def run(*args, quiet_logs=True):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        text = "\n".join(l for l in text.splitlines()
                          if not l.startswith("20") or "[" not in l[:40])
    print(text.strip())
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

try:
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=3) as r:
        print("gateway:", json.load(r)["models"])
except Exception:
    print(f"gateway at {GATEWAY} is NOT answering — start it first:")
    print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


<>:5: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_56/1500538845.py:5: SyntaxWarning: invalid escape sequence '\['
  ANSI = re.compile(chr(27) + "\[[0-9;]*m")


## 1 · The contract suite (12 min)

"Implements the protocol" is a test, not a docstring claim. The same test class is
parametrised over every adapter — and the parametrisation only fills in when the
gateway answers, which is why the count changes with it running.

In [2]:
run("-m", "pytest", "tests/llm/test_adapter_contract.py", "-v", "--no-header", "-q")

................                                                         [100%]
16 passed in 2.21s


0

Four differences between the dialects are load-bearing. Read where each one stops —
this is `AnthropicClient.complete`, and the comment in the middle is the one that
matters most.

In [3]:
import inspect
from murshid.llm.anthropic_client import AnthropicClient
src = inspect.getsource(AnthropicClient.complete)
print(src[:1900])

    def complete(self, request: LLMRequest) -> LLMResponse:
        system, turns = split_system(request.messages)
        t0 = time.perf_counter()
        kwargs: dict = {
            "model": self._route.resolve(request.model_alias),
            "messages": turns,
            "max_tokens": request.max_tokens,  # REQUIRED here, unlike OpenAI-compat
        }
        # A fifth difference, and a live one: as of anthropic 1.x the Messages API
        # no longer takes `temperature` or `top_p` at all — sampling moved behind
        # `output_config.effort`. Forwarding request.temperature here raises a
        # TypeError, which is how this was found. The normalised LLMRequest keeps
        # the field because the OpenAI dialect still has it; the adapter is the
        # right place for the divergence to stop. Do not "fix" this by deleting
        # temperature from LLMRequest, and do not smuggle it through extra_body.
        if system:
            if request.cache_prefix_messages:
      

`temperature` is simply absent — not defaulted, not `None`. The normalised
`LLMRequest` still carries it because the OpenAI dialect still has it; deleting the
field to silence this adapter would be the wrong repair.

## 2 · The open-weight route, with no code change (8 min)

The route already exists in `configs/murshid.yaml`. Read it, then use it.

In [4]:
from murshid.config import get_settings
s = get_settings()
for name, route in s.routes.items():
    print(f"{name:12} {route.base_url:34} residency={route.residency}")

primary      http://gateway:8080/v1             residency=cloud
cheap        http://gateway:8080/v1             residency=cloud
comparison   http://gateway:8080                residency=cloud
vllm         http://gateway:8080/v1             residency=on_premise


In [5]:
run("-m", "murshid.cli", "--route", "vllm", "ask", "ما هي خطوات إصدار سجل تجاري؟")

[service → murshid-onprem via vllm] 749ms, 1674 in (0 cached) / 161 out, 0.334 halalas
بخصوص إصدار سجل تجاري جديد لمؤسسة فردية:
- الرسوم: ٢٠٠ ريال رسوم إصدار و٨٠٠ ريال اشتراك النشاط السنوي
- المدة: يومان من أيام العمل
- المستندات المطلوبة: هوية وطنية سارية، شهادة حجز الاسم التجاري، تسجيل العنوان الوطني
- الخطوات:
  1. احجز الاسم التجاري في البوابة
  2. اختر النشاط من الدليل الوطني للأنشطة
  3. سجّل العنوان الوطني للمنشأة
  4. ادفع رسوم الإصدار واستلم رقم السجل
إذا احتجت مساعدة إضافية يمكنك مراجعة أي مركز من مراكز هيئة الخدمات الحكومية الرقمية، أو الرقم الموحد ١٩٩ على مدار الساعة.
{"route": "vllm", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:30:03.790408Z"}
{"schema": "guard_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "trace_id": "b5ef9a1424

0

Note the `model_id`: `murshid-onprem`. **Nothing in `src/` changed.**
`OpenAICompatClient` works unchanged because vLLM speaks the chat-completions
schema, which is the de-facto wire standard.

## 3 · Streaming and time-to-first-token (12 min)

Record **both** numbers. Then answer, in one sentence: does streaming reduce total
latency?

In [6]:
run("-m", "murshid.cli", "stream", "What documents do I need to renew my commercial registration?")

About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.

[TTFT 315ms · total 389ms · 1406 in (1378 cached) / 138 out]


0

It does not. It reduces *perceived* latency. Generation time is unchanged — which
is why Module 6 treats TTFT and generation as two separate levers with two
different fixes.

Without `stream_options={"include_usage": True}` the usage never arrives and the
cost meter silently undercounts most traffic. The contract test asserts the final
frame carries it:

In [7]:
run("-m", "pytest", "tests/llm/test_adapter_contract.py", "-k", "streaming", "-q", "--no-header")

....                                                                     [100%]


0

## 4 · The fault drill (10 min)

Your instructor may fire this without warning. Fire it yourself here: a 429 storm
with a real `Retry-After` header on the primary model.

In [8]:
print(fault({"mode": "rate_limit", "seconds": 120, "model": "course-flagship", "retry_after": 2}))

{'fault': {'mode': 'rate_limit', 'until': 1788694330.781639, 'model': 'course-flagship', 'retry_after': 2}}


Ask, and keep the log — the retry and the failover *are* the lesson.

In [9]:
run("-m", "murshid.cli", "ask", "كيف أجدد رخصتي التجارية؟", quiet_logs=False)

[faq → murshid-onprem via vllm] 4672ms, 1638 in (0 cached) / 168 out, 0.329 halalas
بخصوص تجديد السجل التجاري:
- الرسوم: ٢٠٠ ريال عن كل سنة تجديد
- المدة: خلال يوم العمل نفسه بعد سداد الرسوم
- المستندات المطلوبة: هوية وطنية أو إقامة سارية، رخصة بلدية سارية، شهادة زكاة للسنة المنتهية
- الخطوات:
  1. سجّل الدخول إلى البوابة برقم الهوية
  2. افتح خدمات الأعمال واختر تجديد السجل التجاري
  3. أكّد النشاط ومدة التجديد
  4. ادفع عبر سداد ونزّل الشهادة المجددة
إذا احتجت مساعدة إضافية يمكنك مراجعة أي مركز من مراكز هيئة الخدمات الحكومية الرقمية، أو الرقم الموحد ١٩٩ على مدار الساعة.
{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:30:12.948496Z"}
{"schema": "guard_verdict", "attempt": 1, "outcome": "first_try", "event": "structured_extracted", "trace_id": "1c5ac9

0

In [10]:
print(fault({"mode": "off"}))

{'fault': {'mode': 'off'}}


Three things to notice:

1. the retry **honoured the header** rather than guessing a backoff — `retry_after`
   travelled from the provider's response into `ResilientClient._delay_for`;
2. attempts were **capped** — retries multiply cost and tail latency;
3. the citizen got an answer, from the on-premise route, and never knew.

What the gateway saw:

In [11]:
stats = gateway_stats()
print(json.dumps({k: stats[k] for k in list(stats)[:8]}, indent=2, ensure_ascii=False))

{
  "requests": 16607,
  "input_tokens": 11429223,
  "output_tokens": 583020,
  "cached_input_tokens": 7311634,
  "cache_hits": 4604,
  "faults_served": 12,
  "by_model": {
    "course-small": 11636,
    "course-anthropic": 62,
    "murshid-onprem": 784,
    "course-flagship": 4125
  },
  "fault": {
    "mode": "off"
  }
}


## 5 · The bench table (8 min)

Twenty bilingual prompts against every configured route, then the token report.

In [12]:
run("scripts/bench_providers.py")

────────────────────────────────────────────────────────────────────────
bench-providers | 20 bilingual prompts x 4 routes
────────────────────────────────────────────────────────────────────────
{"route": "primary", "n": 20, "p50_ms": 104, "p95_ms": 115, "max_ms": 528, "p50_ar_ms": 108, "p50_en_ms": 99, "tokens_in": 30344, "tokens_out": 2686, "cached_input_tokens": 29810, "ar_over_en_input_tokens": 1.16, "sar_total": 0.1908, "halalas_per_call": 0.9539, "residency": "cloud"}
{"route": "cheap", "n": 20, "p50_ms": 50, "p95_ms": 55, "max_ms": 56, "p50_ar_ms": 52, "p50_en_ms": 48, "tokens_in": 30344, "tokens_out": 2686, "cached_input_tokens": 29810, "ar_over_en_input_tokens": 1.16, "sar_total": 0.0081, "halalas_per_call": 0.0407, "residency": "cloud"}
{"route": "comparison", "n": 20, "p50_ms": 122, "p95_ms": 165, "max_ms": 177, "p50_ar_ms": 127, "p50_en_ms": 117, "tokens_in": 30264, "tokens_out": 3186, "cached_input_tokens": 26829, "ar_over_en_input_tokens": 1.16, "sar_total": 0.2482, "hal

0

In [13]:
run("scripts/token_report.py")

────────────────────────────────────────────────────────────────────────
token-report | 100 parallel sentence pairs
────────────────────────────────────────────────────────────────────────
  cl100k_base    en=1022   ar=2373   ar/en=2.32  chars/token en=5.05  ar=1.38  routes: murshid-onprem
  o200k_base     en=1022   ar=1005   ar/en=0.98  chars/token en=5.05  ar=3.26  routes: course-flagship, course-small, course-anthropic

  The same corpus costs 2.32x on cl100k_base and 0.98x on o200k_base.
  Budget per route, not per language. A context-budget check that counts with
  the wrong tokenizer under- or over-budgets by tens of percent — which is the
  whole of the sim-tokenizer-mismatch failure, in one table.


0

Commit the table to `BENCHMARKS.md` with **one sentence**: which route would you
make Murshid's default today, and on what evidence? That sentence is the seed of
the capstone's model-comparison criterion.

Then revise whatever you believed about Arabic. The "costs about twice as much"
rule of thumb is a fact about a *tokenizer generation*; on a current vocabulary it
is gone. The rule that survives is the one underneath: count with the route's own
tokenizer.

## If you finish early

Murshid handles 30,000 conversations a day, averaging six turns, roughly 900 input
and 150 output tokens per turn. At the price sheet's rates, what does routing 70%
of turns to the cheap model save per month? Keep your estimate — Module 6's
`make replay-after` will tell you how wrong it was, and in which direction.